# 1. Drug repositioning

Screen a compound library against one protein target and rank the compounds by
predicted affinity. This is the analysis behind Figure 3 of the paper, run here on a
40-compound subset of the FDA-approved library and a single target (PDE5A).

**Requirements** — the trained weights (`checkpoints/application.pth`, see
`checkpoints/README.md`) and an installed OmniBind (`pip install -e .`).

**Runtime** — about 2 seconds on a laptop CPU.

## 1. Setup

The model runs on the CPU. `cfg.training.device` must be set **before** the model is built,
because the device is read at construction time.

In [ ]:
import os
import sys
import warnings

import pandas as pd
from omegaconf import OmegaConf

warnings.filterwarnings('ignore')

# Run from the repository root so that the relative paths below resolve.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

CHECKPOINT = 'checkpoints/application.pth'
assert os.path.exists(CHECKPOINT), (
    f'{CHECKPOINT} not found. Download the weights from Zenodo first '
    '(see checkpoints/README.md).')

cfg = OmegaConf.load('configs/default.yaml')
cfg.model.type = 'aa3di_gmf'   # must match the checkpoint
cfg.training.device = 'cpu'    # set before build_model/load_model
cfg.training.cuda = False

## 2. Load the model

In [ ]:
from omnibind.predict import load_model

model = load_model(cfg, CHECKPOINT)
print('model loaded:', type(model).__name__)

## 3. Load the target and the compound library

A protein is described by two equally long strings: its amino acid sequence and its
3Di structural-alphabet sequence. `data/sample/sample_proteins.csv` ships both for 100
human proteins; see `data/README.md` for how to generate 3Di sequences for your own
targets with ProstT5.

In [ ]:
proteins = pd.read_csv('data/sample/sample_proteins.csv').set_index('gene_name')
target = proteins.loc['PDE5A']

compounds = pd.read_csv('data/sample/sample_compounds.csv')

print(f"target   : PDE5A ({target['uniprot_id']}), {len(target['aa_sequence'])} residues")
print(f"compounds: {len(compounds)}")
compounds.head()

## 4. Predict

`batch_predict_compounds` runs one forward pass per compound and returns all four
endpoints. Compounds that fail (for example an unparsable SMILES) come back as `None`
rather than raising, so the result is always checked below.

In [ ]:
from omnibind.predict import batch_predict_compounds

results = batch_predict_compounds(
    compounds, target['aa_sequence'], target['3di_sequence'], model, cfg)

failed = results['predicted_ki'].isna().sum()
print(f'predicted {len(results) - failed} / {len(results)} compounds')

## 5. Rank the candidates

Values are on the p-scale (`-log10(M)`), so larger means stronger predicted binding.

In [ ]:
ranked = (results.dropna(subset=['predicted_ki'])
                 .sort_values('predicted_ki', ascending=False)
                 .reset_index(drop=True))
ranked.index += 1
ranked.head(10)[['ID', 'predicted_ki', 'predicted_kd', 'predicted_ic50', 'predicted_ec50']]

The paper ranks candidates by the mean over the four endpoints; the same composite
score is computed here.

In [ ]:
ranked['mean_score'] = ranked[['predicted_ki', 'predicted_kd',
                               'predicted_ic50', 'predicted_ec50']].mean(axis=1)
top = ranked.sort_values('mean_score', ascending=False).head(10)
top[['ID', 'mean_score']]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(range(len(top)), top['mean_score'], color='tab:red')
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['ID'], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('mean predicted score')
ax.set_title('Top candidates for PDE5A')
fig.tight_layout()

## 6. Using your own data

- **Compounds** — any CSV with a `smiles` column. An `id` or `zinc_id` column is used
  as the label; otherwise the row number is used.
- **Target** — supply `aa_sequence` and `3di_sequence` of equal length. The 3Di string
  must contain only the 20 upper-case Foldseek letters.

The same analysis is available as a script:

```bash
python scripts/drug_repositioning.py \
    test.checkpoint_path=./checkpoints/application.pth \
    test.smiles_file=data/sample/sample_compounds.csv \
    test.aa="<target_aa_sequence>" \
    test.sa="<target_3di_sequence>"
```